## Imports

In [2]:
import sys
import os
import torch
import torch.utils.data as data
import torch.nn as nn

sys.path.append(os.path.abspath('../../'))

import neuro_fuzzy_toolbox as nft

In [3]:
import numpy as np

In [4]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

In [5]:
from ucimlrepo import fetch_ucirepo

# Heart Disease

## Model & Training

In [6]:
heart_disease = fetch_ucirepo(id=45)

X = heart_disease.data.features
y = heart_disease.data.targets

In [7]:
X = X.fillna(value=0)

In [8]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        303 non-null    float64
 12  thal      303 non-null    float64
dtypes: float64(3), int64(10)
memory usage: 30.9 KB


In [9]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y)

In [10]:
unique_classes, counts = np.unique(y_train.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1 2 3 4]
[115  38  25  25   9]


In [11]:
unique_classes, counts = np.unique(y_test.values, return_counts=True)
print(unique_classes)
print(counts)

[0 1 2 3 4]
[49 17 11 10  4]


In [12]:
scaler = MinMaxScaler(feature_range=(-1, 1))

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)

In [13]:
x_train = torch.from_numpy(x_train)
x_test = torch.from_numpy(x_test)
y_train = torch.from_numpy(y_train.values).squeeze()
y_test = torch.from_numpy(y_test.values).squeeze()

In [14]:
loader = data.DataLoader(data.TensorDataset(x_train, y_train), batch_size = 16, shuffle = True)
x_train = loader.dataset.tensors[0]
y_train = loader.dataset.tensors[1]
x_train.shape, y_train.shape

(torch.Size([212, 13]), torch.Size([212]))

In [15]:
x_train

tensor([[ 0.5833,  1.0000,  1.0000,  ...,  0.0000, -1.0000, -0.1429],
        [ 0.2500,  1.0000,  1.0000,  ..., -1.0000, -1.0000, -0.1429],
        [-0.0417,  1.0000, -1.0000,  ...,  0.0000, -1.0000,  1.0000],
        ...,
        [-0.2083,  1.0000,  1.0000,  ..., -1.0000, -1.0000, -0.1429],
        [ 0.2500,  1.0000, -1.0000,  ...,  1.0000, -1.0000,  1.0000],
        [ 0.3333,  1.0000,  1.0000,  ...,  0.0000, -0.3333, -0.1429]],
       dtype=torch.float64)

In [16]:
y_train

tensor([2, 0, 0, 0, 2, 0, 3, 3, 4, 1, 1, 1, 0, 2, 0, 0, 2, 0, 4, 0, 3, 3, 1, 0,
        1, 0, 0, 0, 0, 0, 4, 0, 0, 1, 0, 0, 3, 0, 1, 0, 0, 0, 1, 0, 0, 0, 3, 0,
        3, 2, 0, 2, 0, 3, 3, 3, 1, 0, 1, 2, 0, 0, 0, 2, 4, 0, 0, 1, 0, 2, 0, 0,
        2, 0, 1, 0, 0, 0, 0, 1, 3, 3, 0, 0, 0, 2, 1, 3, 1, 0, 1, 0, 3, 0, 3, 2,
        0, 4, 0, 0, 0, 1, 3, 0, 2, 2, 3, 2, 0, 1, 0, 3, 2, 0, 0, 0, 0, 0, 0, 0,
        1, 1, 0, 3, 2, 0, 0, 0, 2, 0, 0, 0, 2, 0, 0, 0, 0, 2, 0, 0, 0, 4, 3, 0,
        0, 0, 1, 0, 0, 4, 0, 0, 2, 0, 0, 3, 1, 1, 3, 1, 1, 0, 1, 0, 3, 0, 0, 1,
        1, 3, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 2, 0, 0, 2, 0, 1, 4, 0, 0, 0,
        0, 2, 1, 0, 1, 1, 0, 1, 0, 0, 3, 0, 2, 0, 1, 0, 0, 0, 0, 4])

## Model & Training

### ANFIS

In [18]:
model = nft.rule_reduced_ANFIS(
    input_size = 13,
    num_mfs = 1,
    outputs = 5,
    output_type='multiclass'
)

model.init_premises(x_train)

### Hybrid Learning Algorithm

In [19]:
loss_fn = nn.functional.cross_entropy
epochs = 50
optimizer = torch.optim.AdamW
params = {'lr': 0.001, 'weight_decay': 0.01}
#optimizer = torch.optim.Adam
#params = {'lr': 0.005}
#optimizer = torch.optim.SGD
#params = {'lr': 0.001, 'momentum': 0.9}

early_stopping = nft.EarlyStopping(
    patience=10, 
    delta=0.01
)

In [20]:
trainer = nft.Hybrid_learning_algorithm(
    epochs=epochs,
    loss_function=loss_fn,
    optimizer=optimizer,
    optimizer_params=params,
    early_stopping=early_stopping
)

### SONFIS

In [21]:
Ngrow = 30
dGrow = 0.8
Nsplit = 30
eSplit = 0.7
Nvanish = 8
lVanish = 3

max_iterations = 50

anfis_trainer = trainer

validation = 0.2
sonfis_early_stopping = nft.EarlyStopping(patience=6, delta=0.01)
last_training_iteration = False

In [22]:
sonfis = nft.alt_SONFIS(
    Ngrow=Ngrow,
    dGrow=dGrow,
    Nsplit=Nsplit,
    eSplit=eSplit,
    Nvanish=Nvanish,
    lVanish=lVanish,
    max_iterations=max_iterations,
    ANFIStrainer=anfis_trainer,
    validation=validation,
    early_stopping=sonfis_early_stopping,
    last_training_iteration=last_training_iteration
)

In [23]:
%time sonfis(model, loader, verbose=True)

Iteration:  0/50 - loss: 2.849416 - validation loss: 2.900042
 -> ANFIS rules: 1

Iteration:  1/50 - loss: 1.250218 - validation loss: 1.312726
 -> ANFIS rules: 2

Iteration:  2/50 - loss: 1.164015 - validation loss: 1.282776
 -> ANFIS rules: 4

Iteration:  3/50 - loss: 1.080775 - validation loss: 1.339576
 -> ANFIS rules: 6

No more updates
Iteration:  4/50 - loss: 1.080775 - validation loss: 1.339576
 -> ANFIS rules: 6


Training finished
 -> ANFIS rules: 6

CPU times: user 9.56 s, sys: 68.7 ms, total: 9.63 s
Wall time: 2.01 s


In [24]:
test_measures = nft.get_measures(model, x_test, y_test)

for measure in test_measures:
    print(measure + ':', test_measures[measure])

Accuracy: 0.6043956043956044
Precision: 0.5099088945242791
Recall: 0.6043956043956044
F1: 0.5486263736263736
Confusion Matrix: [[47  0  2  0  0]
 [10  6  1  0  0]
 [ 3  4  2  2  0]
 [ 3  3  4  0  0]
 [ 0  0  2  2  0]]


In [25]:
train_measures = nft.get_measures(model, x_train, y_train)

for measure in train_measures:
    print(measure + ':', train_measures[measure])

Accuracy: 0.6556603773584906
Precision: 0.6234607502861573
Recall: 0.6556603773584906
F1: 0.6231071587168808
Confusion Matrix: [[107   3   1   2   2]
 [ 23  10   5   0   0]
 [  9   2  10   4   0]
 [  5   3   8   8   1]
 [  2   2   0   1   4]]
